# Glass Identification

In [4]:
# Keep things clean - no annoying warnings
import warnings
warnings.filterwarnings('ignore')

# Importing all requried libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder ,StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score,root_mean_squared_error,mean_absolute_error
import os



In [5]:
glass=pd.read_csv("D:\Machine Learning\Datasets\Cases\Glass_Identification\Glass.csv")

from tqdm import tqdm
X, y = glass.drop("Type", axis=1), glass["Type"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)

In [17]:

from sklearn.ensemble import RandomForestClassifier, StackingClassifier,StackingRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


lr=LogisticRegression()
svm=SVC(kernel='linear')
scaler=StandardScaler()
pipe_svm=Pipeline([('SCL',scaler),('SVM',svm)])
dtc=DecisionTreeClassifier(random_state=25)
rf=RandomForestClassifier(random_state=25)
stack=StackingClassifier(estimators=[('LR',lr),('SVM',pipe_svm),("TREE",dtc)],final_estimator=rf)
stack.fit(X_train,y_train)
y_pred=stack.predict(X_test)


In [10]:
from sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score

print(f1_score(y_test,y_pred,average='macro'))

0.5343101343101343


# Using Pass Through

In [11]:
stack=StackingClassifier(estimators=[('LR',lr),('SVM',pipe_svm),("TREE",dtc)]
                         ,final_estimator=rf,passthrough=True)
stack.fit(X_train,y_train)
y_pred=stack.predict(X_test)

In [12]:
print(f1_score(y_test,y_pred,average='macro'))

0.643903903903904


In [14]:
from lightgbm import LGBMClassifier


l_gbm=LGBMClassifier(random_state=25,verbose=-1)
stack=StackingClassifier(estimators=[('LR',lr),('SVM',pipe_svm),("TREE",dtc)]
                         ,final_estimator=l_gbm,passthrough=True)
stack.fit(X_train,y_train)
y_pred=stack.predict(X_test)

In [15]:
print(f1_score(y_test,y_pred,average='macro'))

0.7594017094017094


# Concrete Strength

In [16]:
concrete=pd.read_csv("D:\Machine Learning\Datasets\Cases\Concrete_Strength\Concrete_Data.csv")

In [21]:
concrete.columns

Index(['Cement', 'Blast', 'Fly', 'Water', 'Superplasticizer', 'Coarse', 'Fine',
       'Age', 'Strength'],
      dtype='object')

In [23]:
from tqdm import tqdm
X, y = concrete.drop("Strength", axis=1), concrete["Strength"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)

## Stacking Regressor

In [27]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor


lr=LinearRegression()
ridge=Ridge()
scaler=StandardScaler()
lasso=Lasso()
elastic=ElasticNet()
dtc=DecisionTreeRegressor(random_state=25)
rf=RandomForestRegressor(random_state=25)
x_gm=XGBRegressor(random_state=25)
l_gbm=LGBMRegressor(random_state=25,verbose=-1)
c_gm=CatBoostRegressor(random_state=25,verbose=False)


In [28]:
final_est=[rf,x_gm,l_gbm,c_gm]
pt=[True,False]
scores=[]
for f in final_est:
    for p in pt:
        stack=StackingRegressor(estimators=[('LR',lr),('RIDGE',ridge),("LASSO",lasso),("EL",elastic),("TREE",dtc)]
                         ,final_estimator=f,passthrough=p)
        stack.fit(X_train,y_train)
        y_pred=stack.predict(X_test)
        scores.append([f,p,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('final_est','passthrough','score'))
df_scores.sort_values('score')

,final_est,passthrough,score
6,<catboost.core.CatBoostRegressor object at 0x0...,True,2.970334
4,"LGBMRegressor(random_state=25, verbose=-1)",True,3.279025
2,"XGBRegressor(base_score=None, booster=None, ca...",True,3.556131
0,RandomForestRegressor(random_state=25),True,3.672427
7,<catboost.core.CatBoostRegressor object at 0x0...,False,3.926216
5,"LGBMRegressor(random_state=25, verbose=-1)",False,4.125863
1,RandomForestRegressor(random_state=25),False,4.202246
3,"XGBRegressor(base_score=None, booster=None, ca...",False,4.521698


In [29]:
final_est=[rf,x_gm,l_gbm,c_gm]
pt=[True,False]
scores=[]
for f in final_est:
    for p in pt:
        stack=StackingRegressor(estimators=[('LR',lr),('RIDGE',ridge),("LASSO",lasso),("EL",elastic),("TREE",dtc)]
                         ,final_estimator=f,passthrough=p)
        stack.fit(X_train,y_train)
        y_pred=stack.predict(X_test)
        scores.append([f,p,r2_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('final_est','passthrough','score'))
df_scores.sort_values('score',ascending=False)

,final_est,passthrough,score
6,<catboost.core.CatBoostRegressor object at 0x0...,True,0.922108
4,"LGBMRegressor(random_state=25, verbose=-1)",True,0.917545
2,"XGBRegressor(base_score=None, booster=None, ca...",True,0.898705
0,RandomForestRegressor(random_state=25),True,0.888299
7,<catboost.core.CatBoostRegressor object at 0x0...,False,0.875840
5,"LGBMRegressor(random_state=25, verbose=-1)",False,0.872690
1,RandomForestRegressor(random_state=25),False,0.864594
3,"XGBRegressor(base_score=None, booster=None, ca...",False,0.846943
